# stage4 第三輪重訓 bi-encoder — 補特徵進 text + 降採樣 + 設施 pair

**跟前兩輪的本質差異(已本機驗證):**
1. 真瓶頸是『特徵線索只在 ce_text、召回用 text(text 0% 有線索)』,非訓練資料量。
2. 解法:補特徵詞進 **召回用的 text 欄位**(+3字,不出分佈)→ 模型召回時看得到線索。
3. e5 離線驗證:補 text 後 4/4 設施隱喻召回改善(報稅 0.43→0.97、飲水機 3x)。
4. 降採樣模板化正樣本(31%→3.7%)騰訊號;台電排除(ce_text 都0%線索=本質結構化)。

**雙 gate(全過才落地):** ab_eval all R@30 ≥ 0.26(不退步) + #99 設施標靶 P@30 提升。


## 0. 環境 — clone main + 裝套件


In [ ]:
!nvidia-smi -L
%cd /content
!rm -rf nchu-edge-rental-ai
!git clone -q https://github.com/eric20041027/nchu-edge-rental-ai.git
%cd nchu-edge-rental-ai
!pip -q install torch transformers onnxruntime onnx tokenizers numpy


## 1. 補特徵線索進 text 欄位(召回用)+ 降採樣 + 設施 pair


In [ ]:
# 1a. 補特徵進 text(飲水機/機車位/可報稅/曬衣場;台電排除)
!python -m pipeline.data_prep.enrich_text_features --write

# 1b. 降採樣模板化重複正樣本(每設施骨架 ≤120)
!python -m pipeline.data_prep.balance_train_data --cap 120

# 1c. 生成設施隱喻 pair(正樣本=富化後真實房源 text)
!python -m pipeline.data_prep.gen_facility_intent_data --per 8


## 2. 合併訓練集(降採樣後 + 設施 pair,append 非取代)


In [ ]:
import json
bal = json.load(open('data/processed/recommendation_train_balanced.json'))
fac = json.load(open('data/processed/facility_intent_train.json'))
merged = bal + fac
json.dump(merged, open('data/processed/train_merged_r3.json','w'), ensure_ascii=False)
print(f'合併: {len(bal)} 降採樣後 + {len(fac)} 設施 pair = {len(merged)} 筆')


## 3. 重訓 bi-encoder(rbt6 teacher)


In [ ]:
!TRAIN_DATA_PATH=data/processed/train_merged_r3.json \
    python -m pipeline.model_training.train_bi_encoder \
    --epochs 3 --batch-size 32 --lr 2e-5 --max-length 64 --bf16 --tf32 \
    --output-dir saved_models/rbt6_bi_encoder_r3


## 4. 蒸餾成 rbt3 student(對齊現役 edge 體積)


In [ ]:
!TRAIN_DATA_PATH=data/processed/train_merged_r3.json \
    python -m pipeline.model_training.distill_bi_encoder \
    --teacher-dir saved_models/rbt6_bi_encoder_r3 \
    --output-dir saved_models/rbt3_bi_encoder_r3 \
    --epochs 3 --batch-size 32 --lr 3e-5 --alpha 0.5 --max-length 64 --bf16 --tf32


## 5. Export onnx + 重算房源向量(用富化後 text,同源)


In [ ]:
STUDENT = 'saved_models/rbt3_bi_encoder_r3'  # 跳過蒸餾改 rbt6_bi_encoder_r3
!python -m pipeline.model_training.export_bi_encoder --saved-dir {STUDENT}
# 房源向量從富化後的 text 重算(step1a 已改 property_data.json)
!python -m pipeline.data_prep.build_property_embeddings --saved-dir {STUDENT}
!python -m pipeline.data_prep.build_property_embeddings --check


## 6. 雙 gate — ab_eval(獨立,不退步)+ #99 設施標靶(進步)


In [ ]:
print('=== Gate A: ab_eval 獨立評估(守 0.26 不退步)===')
!python pipeline/evaluation/eval_vector_vs_rulebased.py || echo 'ab_eval NO-GO'
print()
print('=== Gate B: #99 隱藏含義設施標靶 P@30 ===')
!python pipeline/evaluation/eval_hidden_meaning.py


## 7. Gate 判定 + 打包下載(全過才落地)


In [ ]:
import os, json
sz = os.path.getsize('frontend/models/bi_encoder_dir/bi_encoder_quant.onnx')/1e6
emb = json.load(open('frontend/assets/property_embeddings.json'))
pd = len(json.load(open('frontend/assets/property_data.json')))
print(f'onnx={sz:.1f}MB | embeddings={emb["count"]} vs property_data={pd}')
assert emb['count'] == pd, '向量數 != 房源數!同源錯配'
# 富化後的 property_data 也要一起下載(text 已改,前端需同步)
!tar czf /content/stage4_r3_artifacts.tgz \
    frontend/models/bi_encoder_dir/bi_encoder_quant.onnx \
    frontend/assets/property_embeddings.json \
    frontend/assets/property_data.json
from google.colab import files; files.download('/content/stage4_r3_artifacts.tgz')


## 落地(本機,gate 全過才做)

解壓 artifacts 換進 `frontend/`,**本機獨立複核**:
- `python pipeline/evaluation/eval_vector_vs_rulebased.py` → ab_eval all R@30 ≥ 0.26
- `python pipeline/evaluation/eval_hidden_meaning.py` → 設施標靶 P@30 較重訓前提升
- 前端 preview 親驗『想喝水』『要停機車』實際召回對的房源
- 不盲信 Colab 數字;本機複核 = Colab 宣稱才落地。退步則誠實結案。
